In [ ]:
import os, subprocess, sys, time, glob
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
BASE = os.environ.get("K4_INPUT", "/kaggle/input")
hits = glob.glob(os.path.join(BASE, "**", "k4teacher_post.py"), recursive=True); print(hits)
if not hits: raise SystemExit("k4teacher_post.py not found in the attached dataset")
OUT = os.environ.get("K4_OUT", "/kaggle/working/k4post")
env = dict(os.environ)
env.update({"WIDTH": os.environ.get("K4_WIDTH", "1024"), "DEPTH": "16", "N": os.environ.get("K4_N", "6e7"), "SEED": "0",
            "LAYERS": os.environ.get("K4_LAYERS", "3,7,11,14,15"), "OUT": OUT, "CHUNK": "16384", "POST": "1", "PYTHONUNBUFFERED": "1"})
if os.environ.get("K4_SYNTH", "0") != "1": env["MLP_IDX"] = "0"          # public MLP #0 (its pre-activation teacher exists)
t0 = time.time(); p = subprocess.Popen([sys.executable, hits[0]], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end="", flush=True)
p.wait(); print("exit", p.returncode, "elapsed", time.time() - t0)
print(subprocess.run(["du", "-sh", OUT], capture_output=True, text=True).stdout)
if p.returncode != 0: raise SystemExit(f"teacher failed with exit code {p.returncode}")
